In [5]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [6]:
import math
import os
import os.path
import pickle
import time
import zipfile
from typing import Sequence

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import plotnine as pn
import scipy
import pymc as pm
import pycwt

from shoreshop3_analysis.wavelet import CWT, WCT

In [7]:
shoreshop_path = os.path.expanduser('~/abmurraylab/shoreshop3')
input_cur_path = os.path.join(shoreshop_path, 'inputdata/hindcast_1980_2023/shorelines_and_profiles/FRF_Profiles.zip')
with zipfile.ZipFile(input_cur_path, 'r') as zip:
    with zip.open('FRF_Profiles/00001/shorelinePosAtyFRF1.csv') as file:
        input_duck_cur = pd.read_csv(file)

input_duck_cur['date'] = input_duck_cur['time'].apply(lambda s: s[:10])
input_duck_cur['time'] = pd.to_datetime(input_duck_cur['time'])

input_duck_cur

,time,xFRF,yFRF,latitude,longitude,date
0,1980-10-10 17:45:00,107.839,1.0,33.568228,-85.567133,1980-10-10
1,1981-01-20 19:30:00,96.336,1.0,33.568221,-85.567257,1981-01-20
2,1981-02-04 17:00:00,93.645,1.0,33.568220,-85.567286,1981-02-04
3,1981-02-17 19:20:00,95.056,1.0,33.568220,-85.567271,1981-02-17
4,1981-03-03 16:00:00,102.258,1.0,33.568225,-85.567193,1981-03-03
...,...,...,...,...,...,...
684,2019-09-23 18:27:05,121.974,1.0,33.568236,-85.566981,2019-09-23
685,2019-10-15 18:35:10,119.463,1.0,33.568235,-85.567008,2019-10-15
686,2019-10-25 18:12:58,118.113,1.0,33.568234,-85.567023,2019-10-25
687,2019-11-22 19:32:57,117.331,1.0,33.568234,-85.567031,2019-11-22


In [8]:
input_coastsat_path = os.path.join(shoreshop_path, 'inputdata/NC_CoastSat_smoothed.zip')
with zipfile.ZipFile(input_coastsat_path,  'r') as zip:
    with zip.open('NC_CoastSat_smoothed/usa_NC_0045_0041.csv') as file:
        input_duck_coastsat = pd.read_csv(file, names=('date', 'x'))
input_duck_coastsat

,date,x
0,1984-04-14 15:08:14,26.5
1,1984-06-17 15:09:24,28.4
2,1984-07-03 15:09:35,29.8
3,1984-09-21 15:11:10,25.9
4,1984-10-07 15:11:10,20.9
...,...,...
587,2025-12-03 15:41:25,42.8
588,2025-12-19 15:41:27,32.4
589,2025-12-27 15:41:18,29.3
590,2026-01-04 15:41:25,33.8


In [10]:
submissions_path = os.path.join(shoreshop_path, 'submissions/UserSubmissions')

duck_cur_paths = {
    'CEERD-CHL-Devoe': os.path.join(submissions_path, 'CEERD-CHL-Devoe/mipDuck_1980-2023_waves_simplified_ShoreFor_BruunSLR_ERDCCHL_SRD.csv'),
    'COCollective': os.path.join(submissions_path, 'COCollective/mipDuck_1980-2023_waves_simplified-COCollective.csv'),
    'CoSMoS-COAST-conv_IA': os.path.join(submissions_path, 'CoSMoS-COAST_SV/Duck/CoSMoS_COAST_conv_model_with_interannual/mipDuck_1980-2023_CoSMoS_COAST_conv_model_with_interannual_SV.csv'),
    'CoSMoS-COAST-conv': os.path.join(submissions_path, 'CoSMoS-COAST_SV/Duck/CoSMoS_COAST_conv_model_without_interannual/mipDuck_1980-2023_CoSMoS_COAST_conv_model_without_interannual_SV.csv'),
    'CoSMoS-COAST-dmd_IA': os.path.join(submissions_path, 'CoSMoS-COAST_SV/Duck/CoSMoS_COAST_dmd_model_with_interannual/mipDuck_1980-2023_CoSMoS_COAST_dmd_model_with_interannual_SV.csv'),
    'CoSMoS-COAST-dmd': os.path.join(submissions_path, 'CoSMoS-COAST_SV/Duck/CoSMoS_COAST_dmd_model_without_interannual/mipDuck_1980-2023_CoSMoS_COAST_dmd_model_without_interannual_SV.csv'),
    'CSIRO_nearshore': os.path.join(submissions_path, 'CSIRO_nearshore/mipDuck_1980-2023_CWaCS.csv'),
    'CSIRO_nearshore-smoothed': os.path.join(submissions_path, 'CSIRO_nearshore/mipDuck_1980-2023_CWaCS_smoothed.csv'),
    #'ERDC-CHL-CSHORE': os.path.join(submissions_path, 'ERDC-CHL/CSHORE/mipDuck_1980-2023_CSHORE.csv'),
    #'ERDC-CHL-GENCADE': os.path.join(submissions_path, 'ERDC-CHL/GENCADE/mipDuck_1980-2023_HolzenthalDingGENCADE.csv'),
    # Missing: GEOOCEAN (no CSVs, only netcdf files)
    'ICM-TCN': os.path.join(submissions_path, 'ICM-TCN/mipDuck_1980-2023_ICM-TCN_transect.csv'),
    'IdSOR': os.path.join(submissions_path, 'IdSOR/mipDuck_1980-2023_WaveBulkParameters_IdSOR.csv'),
    'IHCantabriaCMEG-DLinear': os.path.join(submissions_path, 'IHCantabriaCMEG/submission/DLinear/mipDuck_1980-2023-DLinear.csv'),
    'IHCantabriaCMEG-ETSformer': os.path.join(submissions_path, 'IHCantabriaCMEG/submission/ETSformer/mipDuck_1980-2023-ETSformer.csv'),
    'IHCantabriaCMEG-FEDformer': os.path.join(submissions_path, 'IHCantabriaCMEG/submission/FEDformer/mipDuck_1980-2023-FEDformer.csv'),
    'IHCantabriaCMEG-iTransformer': os.path.join(submissions_path, 'IHCantabriaCMEG/submission/iTransformer/mipDuck_1980-2023-iTransformer.csv'),
    'IHCantabriaCMEG-MillerDean': os.path.join(submissions_path, 'IHCantabriaCMEG/submission/MillerDean/mipDuck_1980-2023-MillerDean.csv'),
    'IHCantabriaCMEG-moirai': os.path.join(submissions_path, 'IHCantabriaCMEG/submission/moirai/mipDuck_1980-2023-moirai.csv'),
    'IHCantabriaCMEG-Nonstationary_Transformer': os.path.join(submissions_path, 'IHCantabriaCMEG/submission/Nonstationary_Transformer/mipDuck_1980-2023-Nonstationary_Transformer.csv'),
    'IHCantabriaCMEG-PatchTST': os.path.join(submissions_path, 'IHCantabriaCMEG/submission/PatchTST/mipDuck_1980-2023-PatchTST.csv'),
    'IHCantabriaCMEG-SPADS': os.path.join(submissions_path, 'IHCantabriaCMEG/submission/SPADS/mipDuck_1980-2023-SPADS.csv'),
    'IHCantabriaCMEG-SPADS-Huber': os.path.join(submissions_path, 'IHCantabriaCMEG/submission/SPADS-Huber/mipDuck_1980-2023-SPADS-Huber.csv'),
    'IHCantabriaCMEG-SPADS-Scale': os.path.join(submissions_path, 'IHCantabriaCMEG/submission/SPADS-Scale/mipDuck_1980-2023-SPADS-Scale.csv'),
    'IHCantabriaCMEG-TimeMixer': os.path.join(submissions_path, 'IHCantabriaCMEG/submission/TimeMixer/mipDuck_1980-2023-TimeMixer.csv'),
    'IHCantabriaCMEG-timer_xl': os.path.join(submissions_path, 'IHCantabriaCMEG/submission/timer_xl/mipDuck_1980-2023-timer_xl.csv'),
    'IHCantabriaCMEG-TimesNet': os.path.join(submissions_path, 'IHCantabriaCMEG/submission/TimesNet/mipDuck_1980-2023-TimesNet.csv'),
    'IHCantabriaCMEG-TimeXer': os.path.join(submissions_path, 'IHCantabriaCMEG/submission/TimeXer/mipDuck_1980-2023-TimeXer.csv'),
    'IHCantabriaCMEG-ttm': os.path.join(submissions_path, 'IHCantabriaCMEG/submission/ttm/mipDuck_1980-2023-ttm.csv'),
    'IHCantabriaCMEG-Yates09': os.path.join(submissions_path, 'IHCantabriaCMEG/submission/Yates09/mipDuck_1980-2023-Yates09.csv'),
    'LEGI-ShoreFor': os.path.join(submissions_path, 'LEGI/mipDuck_1980-2023_waves_bulk-LEGIShoreFor.csv'),
    'LEGI-ShoreForeKFcp45': os.path.join(submissions_path, 'LEGI/mipDuck_1980-2023_waves_bulk-LEGIShoreForeKFcp45.csv'),
    'LEGI-ShoreForeKFcp55': os.path.join(submissions_path, 'LEGI/mipDuck_1980-2023_waves_bulk-LEGIShoreForeKFcp55.csv'),
    'mtraboulsi689': os.path.join(submissions_path, 'mtraboulsi689/mipDuck_1980-2023_waves_simplified-MohammadTraboulsi.csv'),
    # Missing: ShEPreMo (48 submissions, don't know which to use)
    'UNSW-WRL-Calcraft': os.path.join(submissions_path, 'UNSW-WRL-Calcraft/mipDuck_1980-2023_MTFB.csv'),
}

print(len(duck_cur_paths))

dfs = []
for model, path in duck_cur_paths.items():
    df = pd.read_csv(path, header=1, names=('date', 'x1', 'x2')).dropna()
    df['model'] = model
    dfs.append(df)

duck_cur = pd.concat(dfs)

def fix_date(date: str) -> str:
    if '/' not in date:
        return date
    m, d, y = list(map(int, date.split('/')))
    return f'{y:04}-{m:02}-{d:02}'

duck_cur['date'] = duck_cur['date'].apply(fix_date)

duck_cur

32


FileNotFoundError: [Errno 2] No such file or directory: '/hpc/home/sln33/abmurraylab/shoreshop3/submissions/UserSubmissions/IHCantabriaCMEG/submission/DLinear/mipDuck_1980-2023-DLinear.csv'

In [ ]:
poorly_normalized_models = set(duck_cur[duck_cur['x1'] < 60]['model'].unique())
poorly_normalized_models

In [ ]:
duck_cur_clean = duck_cur[~duck_cur['model'].isin(poorly_normalized_models)]

In [ ]:
mean_pos = duck_cur_clean.groupby('date').agg(x1=('x1', 'mean'), x2=('x2', 'mean')).reset_index()
mean_pos['date'] = pd.to_datetime(mean_pos['date'])
mean_pos

In [ ]:
(
    pn.ggplot()
    + pn.geom_line(pn.aes('time', 'xFRF', group=1, color='"FRF data"'), data=input_duck_cur)
    + pn.geom_line(pn.aes('date', 'x', group=1, color='"CoastSat data"'), data=input_duck_coastsat)
    + pn.geom_line(pn.aes('date', 'x1', color='"Model data (mean)"'), data=mean_pos)
    + pn.scale_color_manual(values={'FRF data': 'k', 'CoastSat data': 'tab:red', 'Model data (mean)': 'tab:blue'})
    # + pn.geom_line(pn.aes('date', 'x1', group=1), data=duck_cur, color='tab:red')
    + pn.scale_x_date()
    + pn.labs(x='Date', y='Shoreline position (m)', title='Comparison of Shoreline Position Across Data Sources')
    + pn.theme(legend_title=pn.element_blank(), figure_size=(8, 4), dpi=120)
)

In [ ]:
duck_cur_clean.groupby('model').agg(first=('date', 'first'), last=('date', 'last'))

In [ ]:

(
    pn.ggplot(duck_cur_clean)
    + pn.geom_line(pn.aes('date', 'x1', group='model', color='model'), size=0.5, alpha=0.67)
    + pn.geom_line(pn.aes('date', 'x1', group=1), data=mean_pos, color='k', size=4, alpha=0.3)
    + pn.geom_line(pn.aes('time', 'xFRF', group=1), data=input_duck_cur, color='k')
    + pn.scale_x_date()
    + pn.labs(x='Date', y='Shoreline position (m)', title='Shoreline position for all models: Duck 1980–2023 transect 1')
    + pn.theme(legend_position='right', figure_size=(16, 6))
    + pn.guides(color=pn.guide_legend(ncol=2))
    
)


In [ ]:
start = time.perf_counter_ns()

selected_model = duck_cur[duck_cur['model'] == 'mtraboulsi689']

dt = 1.0
wavelet = CWT(selected_model['date'], selected_model['x1'].values, dt=dt, dj=1/12, s0=-1, J=-1, wavelet='morlet', freqs=None)

fig = wavelet.plot(title='Power Spectrum (Morlet): Duck 1980–2023 mtraboulsi')
#fig.set_size_inches(12, 6)
fig.savefig('figures/mtraboulsi-cur.png', dpi=180)

elapsed = time.perf_counter_ns() - start
print(f'{elapsed/1e6:.3f} ms')

fig

In [ ]:
selected_model['date']

In [ ]:
def time_regularize(df: pd.DataFrame, cols: str | Sequence[str], dt: float) -> pd.DataFrame:
    '''
    df should have a "time" column which is a pandas datetime. col is the y-value to interpolate.
    '''

    if isinstance(cols, str):
        cols = (cols, )

    start = df['time'].min()
    range = (df['time'].max() - start).days
    t = np.arange(0, range, dt)

    out_cols = {'time': t}
    in_t = df['time'].apply(lambda time: (time - start).days)
    for col in cols:
        out_cols[col] = np.interp(t, in_t, df[col])

    df_interp = pd.DataFrame(out_cols)
    df_interp['time'] = df_interp['time'].apply(lambda time: start + pd.Timedelta(days=time))
    df_interp['date'] = df_interp['time'].apply(lambda time: time.date().isoformat())

    return df_interp

interp_dt = input_duck_cur['time'].diff().median().days
input_duck_cur_interp = time_regularize(input_duck_cur, 'xFRF', interp_dt)
input_duck_cur_interp

In [ ]:
pn.ggplot(input_duck_cur_interp) + pn.geom_line(pn.aes('time', 'xFRF', group=1)) + pn.scale_x_date()

In [ ]:
obs_wavelet = CWT(input_duck_cur_interp['date'], input_duck_cur_interp['xFRF'].values, dt=interp_dt, dj=1/12, s0=-1, J=-1, wavelet='morlet', freqs=None)

obs_fig = obs_wavelet.plot(title='Power Spectrum (Morlet): Duck 1980–2020 Observations')

#obs_fig.set_size_inches(12, 6)
obs_fig

In [ ]:
comparison_df = input_duck_cur_interp.merge(selected_model, on='date', how='left')
comparison_df

In [ ]:
start = time.perf_counter_ns()
wct = WCT(comparison_df['date'], comparison_df['xFRF'].values, comparison_df['x1'].values, dt=interp_dt, dj=1/12, s0=-1, J=-1, sig=False, significance_level=0.95, wavelet='morlet', normalize=True)
elapsed = time.perf_counter_ns() - start
print(f'{elapsed/1e6:.3f} ms')
# took 7 minutes, 16 seconds with sig=True

In [ ]:
wct_fig = wct.plot(title='Profile 1 coherence between FRF and mtraboulsi')
wct_fig.set_size_inches(12, 6)
wct_fig.savefig('figures/wct.png', dpi=180)
wct_fig

In [ ]:
# plt.close('all')

In [ ]:
((
    pn.ggplot(pd.to_datetime(input_duck_cur['time']).diff().apply(lambda x: x.days).reset_index().dropna())
    + pn.geom_histogram(pn.aes('time'), bins=60)
    + pn.labs(x='Gap Length (Days)', y='Count', title='Duck FRF Profile 1 Gap Length')
) | (
    pn.ggplot(pd.to_datetime(input_duck_cur_interp['time']).diff().apply(lambda x: x.days).reset_index().dropna())
    + pn.geom_histogram(pn.aes('time'), bins=60)
    + pn.labs(x='Gap Length (Days)', y='Count', title='Duck FRF Profile 1 Gap Length (Interpolated)')
)) & pn.theme(figure_size=(10, 4))